# Tokenizer
We started to experiment with `Water Margin`

In [3]:
import requests
import pandas as pd
from pathlib import Path
import json
import regex
import time

In [4]:
pat = regex.compile(
    r"[\p{P}\p{S}]|'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+[\p{L}&&\n^\p{Han}]++|[\p{Han}]{1,2}+|\p{N}{1,3}+| ?[^\s\p{L}\p{N}]++[\r\n]*+|\s++$|\s*[\r\n]|\s+(?!\S)|\s")

In [5]:
def get_locations(corpus):

    DATA = Path("../data")
    DATA.mkdir(exist_ok=True)
    TK_DATA = DATA / "tokenizer"
    TK_DATA.mkdir(exist_ok=True)
    CORPUS_DATA = TK_DATA / corpus
    CORPUS_DATA.mkdir(exist_ok=True)
    return CORPUS_DATA

In [6]:
CORPUS = "wm"
CORPUS_DATA = get_locations(CORPUS)

In [7]:
if CORPUS =="xyj":
    url = "https://raw.githubusercontent.com/tennessine/corpus/refs/heads/master/%E8%A5%BF%E6%B8%B8%E8%AE%B0.txt"

elif CORPUS == "wm":
    url = "https://github.com/tennessine/corpus/raw/refs/heads/master/%E6%B0%B4%E6%B5%92%E4%BC%A0.txt"

In [8]:
def get_text(url):
    if (CORPUS_DATA / "corpus.txt").exists() is False:
        text = requests.get(url).text
        with open(CORPUS_DATA / "corpus.txt", "w") as f:
            f.write(text)
    else:
        text = (CORPUS_DATA / "corpus.txt").read_text()

    return text

In [9]:
text = get_text(url)

In [10]:
text[:100]

'《水浒传》施耐庵\r\n\r\n楔子\u3000张天师祈禳瘟疫\u3000洪太尉误走妖魔\r\n\r\n    \t\t\r\n\r\n    话说大宋仁宗天子在位，嘉佑三年三月三日五更三点，天子驾坐紫哀殿，受百官朝贺。但见：\r\n\r\n    祥云迷'

In [11]:
len(text)

899304

In [12]:
utf8 = list(text.encode())

In [13]:
len(utf8)

2580298

In [14]:
freq = dict()

for c1, c2 in zip(utf8,utf8[1:]):
    freq[(c1, c2)] = freq.get((c1, c2),0) + 1

In [15]:
most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])
most_freq[:5]

[((239, 188), 85790),
 ((188, 140), 63929),
 ((228, 184), 53598),
 ((228, 186), 29524),
 ((227, 128), 28899)]

In [17]:
# code_list = list(utf8)

code_collection = list(list(sub_text.encode("utf8")) for sub_text in pat.findall(text))

codes_start = 0
utf_len = len(list(text.encode("utf8")))
for l in code_collection:

    codes_start = max(max(l), codes_start)

vocab = dict()


def merge(ids, temp_vocab):
    if len(ids) < 2:
        return ids
    new_ids = []
    i = 0
    while i < len(ids):
        if i == len(ids) - 1:
            new_ids.append(ids[i])
            break
        token = temp_vocab.get((ids[i], ids[i + 1]))
        if token is not None:
            new_ids.append(token)
            i += 2
        else:
            new_ids.append(ids[i])
            i += 1
    return new_ids

logs = []

last_ts = time.time()
while True:
    freq = dict()

    for code_list in code_collection:

        if len(code_list) < 2:
            continue

        for c1, c2 in zip(code_list, code_list[1:]):
            freq[(c1, c2)] = freq.get((c1, c2), 0) + 1

    most_freq = sorted(tuple((k, v ) for k,v in freq.items()), key=lambda x: -x[1])

    temp_vocab = dict()
    top_freq_ids1 = set()
    top_freq_ids2 = set()

    for (c1, c2), ct in most_freq:
        if c1 in top_freq_ids2 or c2 in top_freq_ids1 or ct < 5:
            break
        top_freq_ids1.add(c1)
        top_freq_ids2.add(c2)

        codes_start += 1

        temp_vocab[(c1, c2)] = codes_start

        vocab[codes_start] = (c1, c2)

    # print(temp_vocab)

    if len(temp_vocab) == 0:
        print(f"\nFinished Vocab:{len(vocab)} /{ct} / {ct/utf_len:.5f}")
        break

    # print(f"Try to merge {len(temp_vocab)}")
    new_code = codes_start

    if len(vocab) % 100 == 0:
        print(f"V:{len(vocab)}x", end="\t")

    ct = 0

    for i in range(len(code_collection)):
        code_collection[i] = merge(code_collection[i], temp_vocab)
        ct += len(code_collection[i])

    new_ts = time.time()

    logs.append({
        "vocab_len": len(vocab),
        "code_list_len": ct,
        "compression": ct / utf_len,
        "time": new_ts - last_ts,
    })

    last_ts = new_ts
    

V:2300x	V:7400x	V:10300x	V:10400x	V:11300x	V:16400x	V:17600x	
Finished Vocab:17862 /4 / 0.00000


In [18]:
df = pd.DataFrame(logs)
df

,vocab_len,code_list_len,compression,time
0,1,2494508,0.966752,0.794729
1,5,2318558,0.898562,0.747751
2,20,2106831,0.816507,0.696142
3,47,1879590,0.728439,0.613470
4,66,1776162,0.688355,0.595100
...,...,...,...,...
950,17815,548202,0.212457,0.308131
951,17834,548107,0.212420,0.242697
952,17846,548047,0.212397,0.306377
953,17856,547997,0.212377,0.241007


In [21]:
with open(CORPUS_DATA / "vocab.json", "w") as f:
    f.write(json.dumps(vocab, indent=2), )

with open(CORPUS_DATA / "freq.json", "w") as f:
    f.write(json.dumps(most_freq, indent=2))

In [22]:
df = pd.DataFrame(logs)
df.to_csv(str(CORPUS_DATA / "logs.csv"), index=False)

## Retro Analysis

In [23]:
import json
import pandas as pd

In [24]:
df = pd.read_csv(str(CORPUS_DATA / "logs.csv"))

In [25]:
with open(CORPUS_DATA/"vocab.json") as f:
    vocab = dict((int(k), v) for k, v in json.loads(f.read()).items())

with open(CORPUS_DATA/"freq.json") as f:
    most_freq = json.loads(f.read())

In [26]:
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_df(df):
    cols = ["vocab_len", "code_list_len", "compression", "time"]
    titles = ["Vocab Size", "Code List Length", "Compression Ratio", "Time per Step (s)"]

    fig = make_subplots(rows=2, cols=2, subplot_titles=titles)

    for i, (col, title) in enumerate(zip(cols, titles)):
        row, col_idx = divmod(i, 2)
        fig.add_trace(
            go.Scatter(x=df.index, y=df[col], mode="lines", name=title),
            row=row + 1, col=col_idx + 1,
        )

    fig.update_layout(height=600, title_text="BPE Tokenizer Training", showlegend=False)
    fig.show()


In [27]:
plot_df(df)

In [28]:
pair_to_code = dict((tuple(v), int(k)) for k, v in vocab.items())

In [29]:
from loguru import logger

In [30]:
def encode(text: str, pair_to_code):
    code_list = list(text.encode("utf8"))
    if len(code_list) < 2:
        return code_list
    while len(code_list) > 1:
        temp_vocab = dict()
        for c1, c2 in zip(code_list, code_list[1:]):
            token = pair_to_code.get((c1, c2))
            if token is None:
                continue
            else:
                temp_vocab[(c1, c2)] = token
        if len(temp_vocab) == 0:
            return code_list


        pair = min(temp_vocab, key=lambda x: temp_vocab[x])
        token = temp_vocab[pair]

        new_code_list = []

        i = 0
        while i < len(code_list):
            if i < len(code_list) - 1 and code_list[i] == pair[0] and code_list[i + 1] == pair[1]:
                logger.info(f"[REPLACE]{code_list[i]}, {code_list[i + 1]} -> {token}")
                new_code_list.append(token)
                i += 2
            else:
                new_code_list.append(code_list[i])
                i += 1
        
        code_list = new_code_list
            
    return code_list

In [34]:
list("水泊梁山".encode("utf8"))

[230, 176, 180, 230, 179, 138, 230, 162, 129, 229, 177, 177]

In [37]:
token_ids = encode("水泊梁山", pair_to_code=pair_to_code)
print(token_ids)

2026-06-25 20:02:41.576 | INFO     | __main__:encode:25 - [REPLACE]229, 177 -> 334
2026-06-25 20:02:41.577 | INFO     | __main__:encode:25 - [REPLACE]334, 177 -> 356
2026-06-25 20:02:41.578 | INFO     | __main__:encode:25 - [REPLACE]230, 176 -> 392
2026-06-25 20:02:41.579 | INFO     | __main__:encode:25 - [REPLACE]230, 179 -> 468
2026-06-25 20:02:41.579 | INFO     | __main__:encode:25 - [REPLACE]230, 162 -> 502
2026-06-25 20:02:41.579 | INFO     | __main__:encode:25 - [REPLACE]392, 180 -> 543
2026-06-25 20:02:41.580 | INFO     | __main__:encode:25 - [REPLACE]502, 129 -> 624
2026-06-25 20:02:41.580 | INFO     | __main__:encode:25 - [REPLACE]468, 138 -> 703
2026-06-25 20:02:41.581 | INFO     | __main__:encode:25 - [REPLACE]624, 356 -> 743
2026-06-25 20:02:41.581 | INFO     | __main__:encode:25 - [REPLACE]543, 703 -> 3568


[3568, 743]


In [36]:
token_ids = encode("八百里水泊梁山", pair_to_code=pair_to_code)
print(token_ids)

2026-06-25 20:02:35.586 | INFO     | __main__:encode:25 - [REPLACE]229, 133 -> 253
2026-06-25 20:02:35.594 | INFO     | __main__:encode:25 - [REPLACE]233, 135 -> 281
2026-06-25 20:02:35.599 | INFO     | __main__:encode:25 - [REPLACE]281, 140 -> 297
2026-06-25 20:02:35.600 | INFO     | __main__:encode:25 - [REPLACE]229, 177 -> 334
2026-06-25 20:02:35.601 | INFO     | __main__:encode:25 - [REPLACE]334, 177 -> 356
2026-06-25 20:02:35.605 | INFO     | __main__:encode:25 - [REPLACE]230, 176 -> 392
2026-06-25 20:02:35.608 | INFO     | __main__:encode:25 - [REPLACE]231, 153 -> 452
2026-06-25 20:02:35.608 | INFO     | __main__:encode:25 - [REPLACE]230, 179 -> 468
2026-06-25 20:02:35.609 | INFO     | __main__:encode:25 - [REPLACE]230, 162 -> 502
2026-06-25 20:02:35.609 | INFO     | __main__:encode:25 - [REPLACE]392, 180 -> 543
2026-06-25 20:02:35.609 | INFO     | __main__:encode:25 - [REPLACE]452, 190 -> 599
2026-06-25 20:02:35.609 | INFO     | __main__:encode:25 - [REPLACE]502, 129 -> 624
2026

[7971, 297, 3568, 743]


In [38]:
token_ids = encode("诏安", pair_to_code=pair_to_code)
print(token_ids)

2026-06-25 20:02:55.966 | INFO     | __main__:encode:25 - [REPLACE]229, 174 -> 250
2026-06-25 20:02:55.969 | INFO     | __main__:encode:25 - [REPLACE]232, 175 -> 280
2026-06-25 20:02:55.970 | INFO     | __main__:encode:25 - [REPLACE]250, 137 -> 523
2026-06-25 20:02:55.970 | INFO     | __main__:encode:25 - [REPLACE]280, 143 -> 1438


[1438, 523]


In [39]:
encode("我想吃三个西瓜", pair_to_code=pair_to_code)

2026-06-25 20:03:06.218 | INFO     | __main__:encode:25 - [REPLACE]228, 184 -> 242
2026-06-25 20:03:06.223 | INFO     | __main__:encode:25 - [REPLACE]228, 184 -> 242
2026-06-25 20:03:06.228 | INFO     | __main__:encode:25 - [REPLACE]229, 144 -> 266
2026-06-25 20:03:06.231 | INFO     | __main__:encode:25 - [REPLACE]230, 136 -> 279
2026-06-25 20:03:06.232 | INFO     | __main__:encode:25 - [REPLACE]242, 170 -> 285
2026-06-25 20:03:06.233 | INFO     | __main__:encode:25 - [REPLACE]279, 145 -> 315
2026-06-25 20:03:06.239 | INFO     | __main__:encode:25 - [REPLACE]242, 137 -> 368
2026-06-25 20:03:06.241 | INFO     | __main__:encode:25 - [REPLACE]230, 131 -> 457
2026-06-25 20:03:06.243 | INFO     | __main__:encode:25 - [REPLACE]266, 131 -> 544
2026-06-25 20:03:06.250 | INFO     | __main__:encode:25 - [REPLACE]232, 165 -> 636
2026-06-25 20:03:06.251 | INFO     | __main__:encode:25 - [REPLACE]636, 191 -> 664
2026-06-25 20:03:06.253 | INFO     | __main__:encode:25 - [REPLACE]457, 179 -> 872
2026

[6201, 544, 916, 664, 5081]

In [40]:
def decode(tokens, vocab):
    result = list(tokens)
    
    while True:
        i = 0
        new_result = []
        temp_vocab = dict((i, vocab[i]) for i in result if i in vocab)
        if len(temp_vocab) == 0:
            break
        token = max(temp_vocab, key=lambda x: x)
        c1, c2 = temp_vocab[token]
        for t in result:
            if t == token:
                new_result.append(c1)
                new_result.append(c2)
            else:
                new_result.append(t)
        result = new_result
        
    return bytes(result).decode(errors="replace")


def by_token_decode(tokens, vocab):
    results = []
    for token in tokens:
        by_token_str = decode([token], vocab)
        results.append(by_token_str)
    return results
    

In [41]:
for i in range(1000, 2000):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else ", ")

索, 买, 罪, 乱, 原来, 节, 退, 色, 秦明, 却是, 目, 夫, 功, 山寨, 赏, 公孙, 于, 看了, 如今, 玉, �
营, �, 亦, 雷, 归, 饭, 物, 情, 押, 陈, 新, 翻, 廷, 穿, 施, 降, 虽, 蔡, 鼓, 正是, —, 的人, 车, 嫂, 响, 势, 害, 只得, 喝道, 非
所, 宿, 苦, 及, 怒, 撞, 依, 一齐, 公孙胜, 抢, 迁, 张顺, 学, 付, 皆, 史进, 段, 笑道, 便道, 御, 兄长, 仝, 天子, 光, 理, 讨, 朱仝, 大喜, 活, 迳
�, 阮小, 这厮, 乃, 当下, 果, 丈, 认, 牌, 臣, 达, 知府, 院, 关胜, 殿, 张清, 贵, 跟, �, 民, 通, 毕, 商议, 息, 愿, 庄客, 里面, �, 特, �
军士, 大官人, 杨雄, 皇, 香, 挂, 恐, 推, 调, 上山, �, 端, 食, 饮, 弓, 匹, 常, 四个, 城中, 李俊, 灯, 侍, 折, 小弟, 宁, 圣, 一声, 将来, 性命, 宣
弟兄, 猛, 每, 那妇人, 公人, 拾, 董, 前面, 器, 伙, 夺,  宋江, 东京, 树, 结, 徐, 炮, 盘, 番, 不要, 容, 是个, 失, 岸, 下山, 探, 下来, 喊, 乐, 识
件,  林冲, 腰, �, 那个, 杯, �, 摆, 败, 前来, 喽, 喽罗, 封, 在此, 威, 西门, �, 座, 怎地, �, 镇, 许多, 只顾, 逃, 辽, 随即, 烧, 骂, 紧, 公明
旨, 聚, 岭, ‘, 斧, 春, 让, 和尚, ’, 强, 琼, 跳, 护, 寺, 收拾, 珍, 甚么, 周, 闲, 小喽罗, 孔, 语, 服, 皮, 於, 与他, 实, 牛, 那人, 遭
扎, 庄上, 僧, �, 部, 众将, 赐, 济, 答道, �, 也不, 追, 起身, 单, 勇, 兵马, 一条, 阳, 造, 伤, 怪, 高太尉, 庙, 朴, 枢, 助, 参, 兀, 欲, 与我
银子, 田, 宋公明, 攻, 汤, �, 慌忙, 足, 超, 众头领, 暗, 脱, 回来, 犯, 满, �, 斗, 禀, 朴刀, 鸟, 你们, 副, 时迁, 恁, 整, 吃了, 约, �, 麽, 军师
在这里, 观, 朝廷, 当日, 不可, 病, 射, 认得, 席, 胡, �, 空,

In [42]:
for i in range(15000, 15500):
    print(
        decode([i], vocab=vocab),
        end="\n" if i % 30 == 0 else "| ")

马到
两下里| 筛锣| 且战且走| 与本州| 两头蛇解珍| 孙新道| 登云| 出林龙| 竹节| 面带| 面带忧| 面带忧容| 八人| 忙教| 一百余| 早有一| 正东| 只一斧| 人心| 分作三队| 被宋江| 两匹马| 众皆大喜| 不辨| 战袄| 充饥| 那婆婆| 正著| 哀告| 所犯
捧著| 得著| 度岭| 叔叔柴皇城| 李逵和| 金枝| 玉叶| 出屋| 条例| 誓书| 灵位| 拳擦| 拳擦掌| 高廉喝道| 庄客李大| 小校来报| 黑厮| 引马步军兵| 上城守护| 号为| 文宝| 高廉见| 飞沙| 过对阵| 把剑一| 把剑一挥| 此法| 夺路而走| 一齐掩杀| 扎下寨栅
来助| 九宫县| 幼年| 观前| 把上项| 安邦| 棘| 鼍| 鼍鼓| 左手下| 那面| 攻击得紧| 扬尘| 放火为号| 自乱| 救军| 谁人敢| 箩里| 李逵从| 所过州县| 升殿| 加官| 任用| 宣取| 设宴管待| 府前下马| 谢恩已罢| 颍| 关支| 引步军
次日天晓| 也不打话| 舞起双鞭| 逆贼| 一个使| 与呼延灼| 亲解其| 左是| 队军马| 群贼| 名号| 失色| 风火| 连环甲马| 无对| 柏树| 如意| 尽情| 制造| 二停| 宋江便教| 各守| 卵| 怎知| 叔叔孔宾| 犬马之劳| 三千人马| 一棍| 三才| 飞熊
白旄| 不进| 郓城小吏| 聚山林| 擎杯| 内府| 清晨| 长蛇之| 在阵中| 其余的| 炮声| 践| 苦谏| 不闻| 正射| 水浒寨| 嘱咐| 领众| 赏罚| 遵守| 毋得| 吴用笑道| 统领大军| 短箭| 李固和| 不就这里| 右边是| 围在| 张孔目| 司前
望东便走| 寡不敌众| 临阵| 轰天雷凌振| 惧哉| 冲出| 滥官污吏| 随即传令| 声息| 枢密院官| 王府| 青龙偃月刀| 兵书| 已久| 中计| 丑郡马宣赞| 钢枪| 未尝| 巧言| 有计| 关某| 为前部先锋| 探马报道| 梁山泊军马| 用好言抚慰| 疲| 老丈道| 亡过| 正迎著| 叹息
却调| 以防| 都是马军| 八路| 即今| 鬼哭| 杀过来| 金珠细软| 正撞著| 透重围| 大设筵宴| 罪恶| 剜心| 调兵遣将| 群臣| 览| 良臣| 猖�| 猖獗| 酒海| 单延| 圣水将军| 火器| 宋江见报| 彪形大汉| 枯树山| 征袍| 狮蛮| 翡| 二位将军
对儿在阵前| 啧| 辱国| 杀入

Turn down the logging level a little bit

In [43]:
import sys
logger.remove()
logger.add(sys.stderr, level="WARNING")

1

In [55]:
token_ids = encode("山东宋江 河北玉麒麟", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['山东宋江', ' ', '河北', '玉麒麟']

In [56]:
token_ids = encode("景阳冈", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['景阳冈']

In [68]:
token_ids = encode("你若有心，吃我这半盏儿残酒。", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['你若', '有心', '，', '吃', '我这', '半', '盏儿', '残', '酒', '。']

In [69]:

token_ids = encode("hello", pair_to_code=pair_to_code)
by_token_decode(token_ids, vocab=vocab)

['h', 'e', 'l', 'l', 'o']

In [73]:
print(decode(encode(text[30000:31000], pair_to_code), vocab))


    那老儿直拖鲁达到僻静处，说道：“恩人！你好大胆！见今明明地张挂榜文，出一千贯赏钱捉你，你缘何却去看榜？若不是老汉遇见时，却不被做公的拿了？榜上见写着你年甲，貌相，贯址！”

    鲁达道：“酒家不瞒你说，因为你事，就那日回到状元桥下，正迎着郑屠那厮，被酒家三拳打死了，因此上在逃。一到处撞了四五十日，不想来到这里。你缘何不回东京去，也来到这里？”

    金老道：“恩人在上；自从得恩人救了老汉，寻得一辆车子，本欲要回东京去；又怕这厮赶来，亦无恩人在彼搭救，因此不上东京去。随路望北来，撞见一个京师古邻来这里做买卖，就带老汉父女两口儿到这里。亏杀了他，就与老汉女做媒，结交此间一个大财主赵员外，养做外宅，衣食丰足，皆出於恩人。我女儿常常对他孤老说提辖大恩，那个员外也爱刺枪使棒。尝说道：“怎地恩人相会一面，也好。”

    想念如何能彀得见？且请恩人到家过几日，却再商议。”

    鲁提辖便和金老行。

    不得半里到门首，只见老儿揭起帘子，叫道：“我儿，大恩人在此。”

    那女孩儿浓市艳饰。

    从里面出来，请鲁达居中坐了，插烛也似拜了六拜，说道：“若非恩人垂救，怎能彀有今日！”

    拜罢，便请鲁提辖道：“恩人，上楼去请坐。”

    鲁达道：“不须生受，酒家便要去。”

    金老便道：“恩人既到这里，如何肯放你便去！”

    老儿接了杆棒包裹，请到楼上坐定。

    老儿分付道：“我儿，陪侍恩人坐坐，我去安排饭来。”

    鲁达道：“不消多事，随分便好。”

    老儿道：“提辖恩念，杀身难报；量些粗食薄z??A何足挂齿！”

    女子留住鲁达在楼上坐地。

    金老下来叫了家中新讨的小厮，分付那个娅一面烧着火。

    老儿和这小厮上街来买了些鲜鱼，嫩鸡，酿鹅，肥，时新果子之类归来。

    一面开酒，收拾菜蔬，都早摆了。

    搬上楼来，春台上放下三个盏子，三双筷子，铺下菜蔬果子饭等物。

    娅将银酒烫上酒来。

    父女二人轮番把盏，金老倒地便拜。

    鲁提辖道：“老人家，如何恁地下礼？折杀俺也！”

    金老说道：“恩人听禀，前日老汉初到这里，写个红纸牌儿，旦夕一柱香，父女两个兀自


In [70]:

token_ids = encode(text[30000:31000], pair_to_code=pair_to_code)
print("|".join(by_token_decode(token_ids, vocab=vocab)))


|  |  |那老儿|直|拖|鲁达|到|僻静处|，|说道|：|“|恩人|！|你好大胆|！|见今|明明地|张挂|榜文|，|出一|千贯|赏钱|捉你|，|你缘何|却去|看|榜|？|若不是|老汉|遇见|时|，|却不|被|做公的|拿了|？|榜|上|见|写着|你|年甲|，|貌相|，|贯|�|�|！|”|

    |鲁达|道|：|“|酒家|不瞒|你说|，|因为|你|事|，|就|那日|回到|状|元|桥下|，|正迎着|郑屠|那厮|，|被|酒家|三拳|打死了|，|因此上|在逃|。|一|到处|撞|了|四五十日|，|不想|来到这里|。|你|缘|何不|回东京去|，|也|来到这里|？|”|

    |金老|道|：|“|恩人|在上|；|自从|得|恩人|救了|老汉|，|寻|得一|辆车子|，|本|欲要|回东京去|；|又怕|这厮|赶来|，|亦无|恩人|在彼|搭|救|，|因此|不上|东京去|。|随|路|望北|来|，|撞见|一个|京师|古|邻|来这里|做买卖|，|就带|老汉|父女|两口儿|到这里|。|亏|杀了他|，|就与|老汉|女|做媒|，|结|交|此间|一个大|财主|赵员外|，|养|做|外|宅|，|衣|食|丰|足|，|皆|出|於|恩人|。|我女儿|常常|对|他|孤|老|说|提辖|大恩|，|那个|员外|也|爱|刺枪|使棒|。|尝|说道|：|“|怎地|恩人|相会|一面|，|也好|。|”|

    |想念|如何能彀|得见|？|且请|恩人|到家|过几日|，|却再|商议|。|”|

    |鲁提辖|便和|金老|行|。|

    |不得|半里|到|门首|，|只见|老儿|揭起帘子|，|叫道|：|“|我儿|，|大恩人|在此|。|”|

    |那|女|孩儿|浓|市|艳|饰|。|

    |从里面|出来|，|请|鲁达|居中坐了|，|插|烛|也似|拜了|六|拜|，|说道|：|“|若非|恩人|垂|救|，|怎|能彀|有|今日|！|”|

    |拜罢|，|便请|鲁提辖|道|：|“|恩人|，|上楼去|请坐|。|”|

    |鲁达|道|：|“|不须|生受|，|酒家|便要去|。|”|

    |金老|便道|：|“|恩人|既|到这里|，|如何肯|放你|便去|！|”|

    |老儿|接了|杆棒|包裹|，|请到|楼上|坐定|。|

    |老儿|分付道|：|“|我儿|，|陪侍|恩人|坐|坐|，|我去|安排|饭

In [71]:
decode(encode(text[:1000], pair_to_code), vocab) == text[:1000]

True